# Local-to-Cloud Bridge: From ChromaDB to AWS Bedrock

## The Strategic Transition

You've built RAG systems locally. Now scale them to production without vendor lock-in. This module bridges **local open-source architectures** (Ollama + ChromaDB) with **managed cloud services** (AWS Bedrock Knowledge Bases), teaching you to make informed tradeoffs.

## Part 1: Local RAG Architecture (Your Baseline)

### Setup: Ollama + ChromaDB + FastAPI

```bash
# Start Ollama locally
ollama pull mistral:7b
ollama serve

# In another terminal, install dependencies
pip install chromadb fastapi uvicorn requests
```

### Local Embedding + Retrieval



**Cost:** $0 (runs on your machine)
**Latency:** ~500ms per query (depends on hardware)
**Control:** 100% (your data never leaves your infrastructure)

---

In [ ]:
import chromadb
import requests
import json

# Initialize local ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection(name="documents")

# Documents to index
docs = [
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Transformers revolutionized NLP"
]

# Embed locally using Ollama
def embed_with_ollama(text):
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "mistral:7b", "prompt": text}
    )
    return response.json()["embedding"]

# Add to ChromaDB
for i, doc in enumerate(docs):
    embedding = embed_with_ollama(doc)
    collection.add(
        ids=[f"doc_{i}"],
        embeddings=[embedding],
        documents=[doc]
    )

# Query
query = "What is deep learning?"
query_embedding = embed_with_ollama(query)
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)
print(results["documents"])

## Part 2: The Tradeoff Analysis

### Local Architecture Costs

| Dimension | Local (Ollama + ChromaDB) | AWS Bedrock KB |
|-----------|---------------------------|-----------------|
| **Setup Time** | 10 min | 5 min |
| **Monthly Cost** | $0 (your compute) | $0.50/1K docs + $0.10/1K queries |
| **Latency** | 200-1000ms | 50-200ms |
| **Scaling** | Manual (add GPU) | Automatic |
| **Data Privacy** | 100% on-prem | AWS-managed |
| **Customization** | Full control | Limited |

### When to Use Each

**Use Local (Ollama + ChromaDB):**
- Sensitive data (healthcare, finance)
- High query volume (cost-sensitive)
- Custom embedding models
- Offline-first applications

**Use AWS Bedrock KB:**
- Rapid prototyping
- Enterprise compliance requirements
- Multi-model support (Claude, Llama, Mistral)
- Managed scaling

---

## Part 3: Migrating to AWS Bedrock Knowledge Bases

### Step 1: Create a Knowledge Base



### Step 2: Ingest Documents



### Step 3: Query the Knowledge Base



---

In [ ]:
import boto3

bedrock_agent = boto3.client('bedrock-agent', region_name='us-east-1')

# Create knowledge base
response = bedrock_agent.create_knowledge_base(
    name='my-rag-kb',
    description='Production RAG system',
    roleArn='arn:aws:iam::ACCOUNT:role/BedrockKBRole',
    knowledgeBaseConfiguration={
        'type': 'VECTOR',
        'vectorKnowledgeBaseConfiguration': {
            'embeddingModelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-embed-text-v1'
        }
    },
    storageConfiguration={
        'type': 'OPENSEARCH',
        'opensearchServerlessConfiguration': {
            'collectionArn': 'arn:aws:aoss:us-east-1:ACCOUNT:collection/...'
        }
    }
)

kb_id = response['knowledgeBase']['id']
print(f"Knowledge Base ID: {kb_id}")

In [ ]:
# Upload documents to S3
import boto3
s3 = boto3.client('s3')

with open('documents.pdf', 'rb') as f:
    s3.upload_file('documents.pdf', 'my-kb-bucket', 'documents.pdf')

# Trigger ingestion
response = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId='data-source-id'
)

print(f"Ingestion job: {response['ingestionJob']['ingestionJobId']}")

In [ ]:
# Retrieve relevant documents
response = bedrock_agent.retrieve(
    knowledgeBaseId=kb_id,
    retrievalConfiguration={
        'vectorSearchConfiguration': {
            'numberOfResults': 5
        }
    },
    retrievalQuery={
        'text': 'What is deep learning?'
    }
)

# Use with Claude for generation
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

context = "\n".join([r['content']['text'] for r in response['retrievalResults']])

response = bedrock_runtime.invoke_model(
    modelId='anthropic.claude-3-sonnet-20240229-v1:0',
    body=json.dumps({
        'anthropic_version': 'bedrock-2023-06-01',
        'max_tokens': 1024,
        'system': f'Use this context: {context}',
        'messages': [
            {'role': 'user', 'content': 'What is deep learning?'}
        ]
    })
)

print(response['body'].read().decode())

## Part 4: Hybrid Architecture (Best of Both)

For maximum flexibility, run **local embeddings** with **AWS Bedrock generation**:



**Benefits:**
- Embeddings stay local (privacy + cost)
- Retrieval scales on AWS
- Generation uses best-in-class models
- Pay only for what you use

---

In [ ]:
# Embed locally (fast, private)
local_embedding = embed_with_ollama(query)

# Retrieve from AWS (managed, scalable)
response = bedrock_agent.retrieve(
    knowledgeBaseId=kb_id,
    retrievalConfiguration={
        'vectorSearchConfiguration': {
            'numberOfResults': 5,
            'overrideSearchType': 'HYBRID'  # Combine vector + keyword
        }
    },
    retrievalQuery={'text': query}
)

# Generate with Claude (state-of-the-art)
response = bedrock_runtime.invoke_model(
    modelId='anthropic.claude-3-opus-20240229-v1:0',
    body=json.dumps({...})
)

## Key Concepts

| Concept | Local | Cloud |
|---------|-------|-------|
| **Embedding** | Ollama (7B model) | Amazon Titan Embeddings |
| **Storage** | ChromaDB (SQLite) | OpenSearch Serverless |
| **Retrieval** | Vector similarity | Hybrid (vector + keyword) |
| **Generation** | Ollama (7B) | Claude 3 Opus |

---

## Quizzes

### Quiz 1: Cost Analysis
**Question:** You have 100K documents and 10K queries/month. Which is cheaper?
- A) Local (Ollama + ChromaDB) ✓
- B) AWS Bedrock KB
- C) They cost the same
- D) Depends on document size

### Quiz 2: Latency Tradeoff
**Question:** AWS Bedrock KB is faster than local Ollama because:
- A) Managed infrastructure + GPU optimization ✓
- B) AWS has better algorithms
- C) Network latency is lower
- D) It's not actually faster

### Quiz 3: Hybrid Architecture
**Question:** Why embed locally but retrieve from AWS?
- A) Keep data private while leveraging managed infrastructure ✓
- B) Reduce AWS costs
- C) Improve accuracy
- D) Simplify deployment

---

## Resources & References

- **[AWS Bedrock Knowledge Bases](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html)** - Official documentation
- **[Ollama Documentation](https://github.com/ollama/ollama)** - Local LLM inference
- **[ChromaDB](https://www.trychroma.com/)** - Vector database
- **[OpenSearch Serverless](https://docs.aws.amazon.com/opensearch-service/latest/developerguide/serverless.html)** - Managed vector storage